In [1]:
# Author = Prabhu Vellaisamy

# TNN Column Submodule VerilogGen library for Verilog RTL creation
# Original Verilog files created by Harideep Nair 

from veriloggen import *
import numpy as np
import os

In [3]:
class TNN_Functions():
    
    # inhibit operator
    def Less_equal(self):
        
        m = Module('less_equal')
        
        # input-output ports
        data_in = m.Input('data_in', 1)
        inhibit_in = m.Input('inhibit_in', 1)
        aclk = m.Input('aclk', 1)
        rst = m.Input('rst', 1)
        out = m.Output('out', 1)
        
        temp1 = m.Wire('temp1', 1)
        temp2 = m.Wire('temp2', 1)
        
        # submodule
        pulse, pulse_clk_name = self.Pulse2edge()
        
        pulse_inst = m.Instance(pulse, 'pulse_inst', params = None, ports = [aclk, temp1, rst, temp2])
        
        temp1.assign(~data_in & inhibit_in)
        out.assign(data_in & ~temp2)
        
        return m, ('aclk')
    

In [5]:
    # pulse -> edge converter
    def Pulse2edge(self):
        m = Module('pulse2edge')
        
        # input-output ports
        aclk = m.Input('aclk', 1)
        pulse_in = m.Input('pulse_in', 1)
        grst = m.Input('grst', 1)
        edge_out = m.Output('edge_out', 1)
        
        temp = m.Reg('temp', 1)
        edge_out.assign(pulse_in|temp)
        
        # always block
        m.Always(Posedge(aclk)) (
            If(grst) (temp(Int(0, width = 1, base = 2))) 
            .Else (temp(edge_out)))
            
        return m, ('aclk')
        
        

In [6]:
    # adder module
    
    def Adder(self, res = 4):
        m = Module('adder')
        res = m.Parameter('RES', res)
        
        # input-output ports
        a = m.Input('a', res)
        b = m.Input('b', res)
        cin = m.Input('cin')
        out = m.Output('out')
        
        out.assign(a+b+cin)
        
        # no clocks, combinational design
        return m, None
        

In [7]:
    # edge -> pulse converter
    
    def Edge2pulse(self):
        
        m = Module('edge2pulse')
        
        # input-output ports
        edge_in = m.Input('edge_in', 1)
        clk_in = m.Input('clk_in', 1)
        pulse_out = m.Output('pulse_out', 1)
        
        temp1 = m.Reg('temp1', 1)
        temp2 = m.Reg('temp2', 1)
        
        # always block
        m.Always(Posedge(clk_in)) ( 
            temp1(edge_in),
            temp2(temp1)
            )
        
        pulse_out.assign(edge_in & ~temp2)
        return m, ('clk_in')
        

In [8]:
    # increment/decrement logic for synaptic weight updates
    
    def Incdec(self):

        m = Module('incdec')
        
        # input-output ports
        cases = m.Input('stdp_cases', 4)
        capture = m.Input('capture', 1)
        minus = m.Input('minus', 1)
        search = m.Input('search', 1)
        backoff = m.Input('backoff', 1)
        min_case = m.Input('min', 1)
        F = m.Input('F', 1)
        inc = m.Output('inc', 1)
        dec = m.Output('dec', 1)

        temp = m.Wire('temp', 1)

        temp.assign(F | min_case)
        
        inc.assign((cases[0] & capture & temp) | (cases[2] & search))
        dec.assign((cases[1] & minus & temp) | (cases[3] & backoff & temp));
        
        # no clocks returned
        return m, None